# Repeated `idcontrato` in the raw workbooks

`bulk-ingest` keeps one row per `idcontrato` (last wins), so `ingested < read`. This notebook looks
only at the repeated ids in `data/raw/contracts_raw_<YEAR>.parquet`: how many, what differs
between the copies, and what is lost by keeping one.

In [1]:
from pathlib import Path

import polars as pl

from base_gov import raw_ingest

DATA = Path("..") / "data"
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(80)

years = raw_ingest.raw_years(DATA / "raw")
raw = pl.concat(
    [pl.read_parquet(raw_ingest.raw_path(DATA / "raw", y)).with_columns(src_year=pl.lit(y)) for y in years],
    how="diagonal_relaxed",
)
dups = raw.filter(pl.col("idcontrato").is_duplicated()).sort("idcontrato")
print(f"years={years}  raw rows={raw.height:,}  repeated rows={dups.height:,}  ids={dups['idcontrato'].n_unique():,}")

years=[2024, 2025, 2026]  raw rows=645,686  repeated rows=4,264  ids=2,126


## 1. How many, and how many copies per id

In [2]:
per_year = raw.group_by("src_year").agg(
    pl.len().alias("rows"),
    pl.col("idcontrato").is_duplicated().sum().alias("rows_in_repeated_ids"),
    pl.col("idcontrato").filter(pl.col("idcontrato").is_duplicated()).n_unique().alias("repeated_ids"),
).sort("src_year")
display(per_year)
display(dups.group_by("idcontrato").len()["len"].value_counts().sort("len"))
# Do any ids repeat ACROSS years (i.e. would `keep=last` pick by file order)?
cross = dups.group_by("idcontrato").agg(pl.col("src_year").n_unique().alias("n_years")).filter(pl.col("n_years") > 1)
print("ids spanning more than one year file:", cross.height)

src_year,rows,rows_in_repeated_ids,repeated_ids
i32,u32,u32,u32
2024,227111,2118,1059
2025,247334,684,342
2026,171241,1462,725


len,count
u32,u32
2,2120
4,6


ids spanning more than one year file: 0


## 2. Which columns differ between the copies

In [3]:
value_cols = [c for c in raw.columns if c not in ("idcontrato", "src_year")]
differing = dups.group_by("idcontrato").agg([pl.col(c).n_unique().alias(c) for c in value_cols])
diff_counts = (
    differing.select([(pl.col(c) > 1).sum().alias(c) for c in value_cols])
    .transpose(include_header=True, column_names=["ids_where_it_differs"])
    .filter(pl.col("ids_where_it_differs") > 0)
    .sort("ids_where_it_differs", descending=True)
)
diff_counts

column,ids_where_it_differs
str,u32
"""ContratEcologico""",1399
"""TipoCriterioAdjudicacao""",733


## 3. Side-by-side examples\n\nFirst cases where only `TipoCriterioAdjudicacao` differs.

In [4]:
only_criterio = differing.filter(
    pl.all_horizontal([(pl.col(c) == 1) for c in value_cols if c != "TipoCriterioAdjudicacao"])
    & (pl.col("TipoCriterioAdjudicacao") > 1)
)["idcontrato"]
print("ids that differ ONLY in TipoCriterioAdjudicacao:", only_criterio.len())
cols = ["idcontrato", "src_year", "objectoContrato", "adjudicante", "precoContratual", "tipoprocedimento", "TipoCriterioAdjudicacao"]
for cid in only_criterio.head(5):
    display(dups.filter(pl.col("idcontrato") == cid).select(cols))

ids that differ ONLY in TipoCriterioAdjudicacao: 727


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
10611312,2024,"""Fornecimento de Produtos de panificação, produtos frescos de pastelaria e bolos …","""600083314 - Agrupamento de Escolas Manuel Teixeira Gomes, Portimão""",32252.5,"""Consulta Prévia""","""Monofator"""
10611312,2024,"""Fornecimento de Produtos de panificação, produtos frescos de pastelaria e bolos …","""600083314 - Agrupamento de Escolas Manuel Teixeira Gomes, Portimão""",32252.5,"""Consulta Prévia""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
10616271,2024,"""Acordo-Quadro para Fornecimento de Combustíveis Rodoviários” da Central de Compr…","""506833224 - Município de Vila Real de Santo António""",489000.0,"""Ao abrigo de acordo-quadro (art.º 259.º)""","""Multifator"""
10616271,2024,"""Acordo-Quadro para Fornecimento de Combustíveis Rodoviários” da Central de Compr…","""506833224 - Município de Vila Real de Santo António""",489000.0,"""Ao abrigo de acordo-quadro (art.º 259.º)""","""Monofator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
10625543,2024,"""2524000119 Matérias Primas Relojoaria - Protocolo IEFP""","""501390642 - CASA PIA DE LISBOA, I.P. """,1777.0,"""Consulta Prévia""","""Monofator"""
10625543,2024,"""2524000119 Matérias Primas Relojoaria - Protocolo IEFP""","""501390642 - CASA PIA DE LISBOA, I.P. """,1777.0,"""Consulta Prévia""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
10650159,2024,"""Aquisição Viatura Passageiros""","""501413197 - Universidade do Porto""",48949.21,"""Consulta Prévia""","""Monofator"""
10650159,2024,"""Aquisição Viatura Passageiros""","""501413197 - Universidade do Porto""",48949.21,"""Consulta Prévia""","""Multifator"""


idcontrato,src_year,objectoContrato,adjudicante,precoContratual,tipoprocedimento,TipoCriterioAdjudicacao
i64,i32,str,str,f64,str,str
10656968,2024,"""Aquisição de Combustível Rodoviário para a ARM - 2024/2027""","""509574513 - A. R. M. - Águas e Resíduos da Madeira, S. A.""",1.491879e6,"""Concurso público""","""Monofator"""
10656968,2024,"""Aquisição de Combustível Rodoviário para a ARM - 2024/2027""","""509574513 - A. R. M. - Águas e Resíduos da Madeira, S. A.""",1.491879e6,"""Concurso público""","""Multifator"""


## 4. The `TipoCriterioAdjudicacao` pairs\n\nIf each copy carries one award criterion, the pair is a multi-criteria contract split over rows.

In [5]:
pairs = (
    dups.sort("idcontrato", "TipoCriterioAdjudicacao", nulls_last=True)
    .group_by("idcontrato", maintain_order=True)
    .agg(pl.col("TipoCriterioAdjudicacao").alias("criterios"))
    .with_columns(pair=pl.col("criterios").list.join("  |  "))
)
pairs["pair"].value_counts(sort=True).head(15)

pair,count
str,u32
"""Monofator | Monofator""",956
"""Monofator | Multifator""",727
"""Multifator | Multifator""",384
"""""",49
"""Monofator | Monofator | Multifator | Multifator""",6
"""Preço ou custo | Preço ou custo""",3
"""Melhor relação qualidade – preço | Melhor relação qualidade – preço""",1


## 5. Other differing columns (`ContratEcologico`, ...)

In [6]:
other = [c for c in diff_counts["column"] if c != "TipoCriterioAdjudicacao"]
print("other differing columns:", other)
if other:
    ids = differing.filter(pl.any_horizontal([pl.col(c) > 1 for c in other]))["idcontrato"]
    display(dups.filter(pl.col("idcontrato").is_in(ids.head(5).implode())).select("idcontrato", "src_year", *other, "precoContratual"))

other differing columns: ['ContratEcologico']


idcontrato,src_year,ContratEcologico,precoContratual
i64,i32,str,f64
10424261,2024,"""Sim""",3918.75
10424261,2024,"""Não""",3918.75
10424271,2024,"""Não""",467.95
10424271,2024,"""Sim""",467.95
10424433,2024,"""Sim""",27849.0
10424433,2024,"""Não""",27849.0
10424599,2024,"""Não""",20340.0
10424599,2024,"""Sim""",20340.0
10424773,2024,"""Sim""",1069.96


## 6. Impact\n\nWhat keeping *all* copies would do to counts and value, versus `keep=last`.

In [7]:
kept = raw.unique(subset="idcontrato", keep="last", maintain_order=True)
dropped = raw.height - kept.height
double_counted = dups.height - dups["idcontrato"].n_unique()
value_all = dups["precoContratual"].sum()
value_kept = kept.filter(pl.col("idcontrato").is_in(dups["idcontrato"].unique().implode()))["precoContratual"].sum()
print(f"rows dropped by keep=last: {dropped:,} (== extra copies: {double_counted:,})")
print(f"value on repeated ids: all copies €{value_all:,.0f} vs one copy €{value_kept:,.0f} "
      f"(double count €{value_all - value_kept:,.0f})")

rows dropped by keep=last: 2,138 (== extra copies: 2,138)
value on repeated ids: all copies €726,293,845 vs one copy €361,816,302 (double count €364,477,544)


## 7. Reading

_To fill in after review:_ is the extra row a multi-criteria split (lossless to collapse if the
criteria are merged into one cell) or a genuine duplicate? If the former, prefer aggregating
`TipoCriterioAdjudicacao` per `idcontrato` over `keep="last"`, which today discards one criterion
arbitrarily.